# 02 Model Scientific Interest

Canonical modeling notebook for yearly scientific interest.
This notebook computes SI, ranks future high-interest pathogen-antibiotic pairs, and generates focused plots.


In [ ]:
from pathlib import Path

import pandas as pd

from amr.pipelines.scientific_interest import (
    build_yearly_si_dataset,
    rank_future_high_interest_pairs,
)
from amr.scientific_interest import plot_yearly_si_pair


## Configuration


In [ ]:
YEARLY_SCREENED_PATH = Path("yearly_amr_screened_pairs.parquet")
PATHOGEN_BG_PATH = Path("pathogen_tiab_counts.parquet")
ANTIBIOTIC_BG_PATH = Path("antibiotic_tiab_counts.parquet")

YEARLY_SI_PATH = Path("yearly_si.parquet")
RANK_OUTPUT_PATH = Path("yearly_si_ranked_future.parquet")
INTERESTING_OUTPUT_PATH = Path("yearly_si_ranked_future_interesting.parquet")
INTERESTING_PLOT_DIR = Path("interesting_yearly_si_pngs")

FIT_START_YEAR = 2020
FIT_END_YEAR = 2026
NUM_FUTURE_YEARS = 20

MIN_CURRENT_PREDICTED_COUNT = 10
MIN_FUTURE_PREDICTED_COUNT = 10
MIN_SI_CHANGE = 0.1
MIN_POST_FIT_OBSERVED_YEARS = 1
MAX_POST_FIT_SI_ABS_PCT_ERROR_MEAN = 40.0
FALLBACK_TOP_N = 0


## Load Canonical Inputs


In [ ]:
yearly_screened_df = pd.read_parquet(YEARLY_SCREENED_PATH)
pathogen_bg_df = pd.read_parquet(PATHOGEN_BG_PATH)
antibiotic_bg_df = pd.read_parquet(ANTIBIOTIC_BG_PATH)

print(f"yearly_screened rows: {len(yearly_screened_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")


## Build Yearly SI Dataset


In [ ]:
si_df, pooled_model, design_columns = build_yearly_si_dataset(
    yearly_screened_df,
    pathogen_bg_df,
    antibiotic_bg_df,
    output_path=YEARLY_SI_PATH,
)

print(f"SI rows: {len(si_df):,}")
print(f"SI pairs: {si_df[['pathogen','antibiotic']].drop_duplicates().shape[0]:,}")
si_df.head(10)


## Rank Reliable Future High-Interest Pairs


In [ ]:
rank_df, interesting_df = rank_future_high_interest_pairs(
    si_df,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    min_current_predicted_count=MIN_CURRENT_PREDICTED_COUNT,
    min_future_predicted_count=MIN_FUTURE_PREDICTED_COUNT,
    min_si_change=MIN_SI_CHANGE,
    min_post_fit_observed_years=MIN_POST_FIT_OBSERVED_YEARS,
    max_post_fit_si_abs_pct_error_mean=MAX_POST_FIT_SI_ABS_PCT_ERROR_MEAN,
    fallback_top_n=FALLBACK_TOP_N,
    rank_output_path=RANK_OUTPUT_PATH,
    interesting_output_path=INTERESTING_OUTPUT_PATH,
    verbose=True,
)

interesting_df[
    [
        "pathogen",
        "antibiotic",
        "interest_score",
        "si_change",
        "current_predicted_count",
        "future_predicted_count",
        "post_fit_si_abs_pct_error_mean",
    ]
].head(30)


## Plot Top Interesting Pairs


In [ ]:
INTERESTING_PLOT_DIR.mkdir(parents=True, exist_ok=True)

for row in interesting_df.head(25).itertuples(index=False):
    result = plot_yearly_si_pair(
        si_df=si_df,
        pathogen=row.pathogen,
        antibiotic=row.antibiotic,
        fit_start_year=FIT_START_YEAR,
        fit_end_year=FIT_END_YEAR,
        num_additional_years=NUM_FUTURE_YEARS,
        loess_frac=0.6,
        loess_degree=2,
        tail_damping_years=None,
    )
    pathogen_slug = str(row.pathogen).replace("/", "_").replace(" ", "_")
    antibiotic_slug = str(row.antibiotic).replace("/", "_").replace(" ", "_")
    out_path = INTERESTING_PLOT_DIR / f"{pathogen_slug}__{antibiotic_slug}.png"
    result["figure"].savefig(out_path, dpi=200, bbox_inches="tight")

print(f"Saved {min(25, len(interesting_df))} plots to {INTERESTING_PLOT_DIR}")
